<a href="https://colab.research.google.com/github/PeachTj/BranchingExercise/blob/master/translateSRT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!pip install requests pypinyin


In [43]:
!pip install -U google-cloud-translate pypinyin
import os, re, html, time
from pathlib import Path
from typing import List, Tuple

from pypinyin import pinyin, Style
from google.cloud import translate_v3

In [56]:
from google.colab import auth
auth.authenticate_user()  # 弹窗登录你的 Google 账号

In [61]:
import os

os.environ["GOOGLE_CLOUD_PROJECT"] = "888506319622"  # 例: avian-light-471713-r0 或 888506319622
os.environ["GCP_LOCATION"] = "global"                   # 或 "asia-northeast1"（东京）

print("PROJECT =", os.environ["GOOGLE_CLOUD_PROJECT"])
print("LOCATION =", os.environ["GCP_LOCATION"])


PROJECT = 888506319622
LOCATION = global


In [63]:
! gcloud auth application-default login


You are running on a Google Compute Engine virtual machine.
The service credentials associated with this virtual machine
will automatically be used by Application Default
Credentials, so it is not necessary to use this command.

If you decide to proceed anyway, your user credentials may be visible
to others with access to this virtual machine. Are you sure you want
to authenticate with your personal account?

Do you want to continue (Y/n)?  y

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=sIIOZ4C8XH46s8wUGpYVBLLZROkHwl&prompt=consent&token_

In [65]:
! gcloud auth application-default set-quota-project avian-light-471713-r0


Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "avian-light-471713-r0" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [66]:
# 兼容导入：优先使用 translate_v3；若不可用，则回退到 translate
try:
    from google.cloud import translate_v3
except ImportError:
    from google.cloud import translate as translate_v3

import html

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
LOCATION   = os.environ.get("GCP_LOCATION", "global")

# 最小自检：翻译“你好”到日语，验证凭据 + 项目 + API 都没问题
def translate_once_v3(text="你好", src="zh-CN", tgt="ja"):
    client = translate_v3.TranslationServiceClient()
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"
    resp = client.translate_text(
        request={
            "parent": parent,
            "contents": [text],
            "mime_type": "text/plain",
            "source_language_code": src,
            "target_language_code": tgt,
        }
    )
    out = [html.unescape(t.translated_text) for t in resp.translations]
    print("✅ translate_once_v3:", out[0])
    return out[0]

_ = translate_once_v3()  # 看到日语“こんにちは”即表示通过


✅ translate_once_v3: こんにちは


In [67]:
from google.colab import files
uploaded = files.upload()   # 选择你的 .srt 文件
list(uploaded.keys())       # 显示已上传文件名


Saving [zmk.pw][哪吒闹海]1979 1080p CHN Blu-ray AVC LPCM 1.0-AREY1.srt to [zmk.pw][哪吒闹海]1979 1080p CHN Blu-ray AVC LPCM 1.0-AREY1 (1).srt


['[zmk.pw][哪吒闹海]1979 1080p CHN Blu-ray AVC LPCM 1.0-AREY1 (1).srt']

In [68]:
import re, time
from pathlib import Path
from typing import List, Tuple
from pypinyin import pinyin, Style

SOURCE_LANG = "zh-CN"
TARGET_LANG = "ja"
TONE_TYPE   = "mark"          # "mark" → lǎo yé；"number" → lao3 ye2

# 分批阈值：避免 400（过大）
MAX_ITEMS_PER_BATCH = 100     # 每批条数（v3 建议控制到 ~100）
MAX_CHARS_PER_BATCH = 20000   # 每批字符总量（Advanced ≤~30k code points，留余量）
SLEEP_BETWEEN_BATCH = 0.15    # 轻微限速，避免 429

def clean_text(line: str) -> str:
    """去除 { ... } 样式与控制字符，仅保留可翻译文字"""
    line = re.sub(r"\{.*?\}", "", line)
    line = re.sub(r"[\u0000-\u001F]", "", line)
    return line.strip()

def is_dialogue(line: str) -> bool:
    """对白行（非编号、非时间轴、非空白）；同时处理带 BOM 的编号行"""
    clean = line.strip().lstrip("\ufeff")
    if not clean:         return False
    if clean.isdigit():   return False          # 编号
    if "-->" in clean:    return False          # 时间轴
    return True

def to_pinyin(text: str, tone_type="mark") -> str:
    """拼音（带声调或数字调）"""
    style = Style.TONE if tone_type == "mark" else Style.TONE3
    return " ".join(s[0] for s in pinyin(text, style=style))

def batch_iter(lines: List[str],
               max_items=MAX_ITEMS_PER_BATCH,
               max_chars=MAX_CHARS_PER_BATCH):
    """把待翻译文本按“条数+总字数”双重限制切批"""
    cur, cur_chars = [], 0
    for s in lines:
        l = len(s)
        # 单条过长：强制切块（极少见）
        if l > max_chars:
            for i in range(0, l, max_chars - 1000):
                piece = s[i:i+max_chars-1000]
                if cur and (len(cur) >= max_items or cur_chars + len(piece) > max_chars):
                    yield cur; cur, cur_chars = [], 0
                cur.append(piece); cur_chars += len(piece)
            continue

        if cur and (len(cur) >= max_items or cur_chars + l > max_chars):
            yield cur; cur, cur_chars = [], 0
        cur.append(s); cur_chars += l
    if cur: yield cur

def translate_batch_v3(client: translate_v3.TranslationServiceClient,
                       project_id: str,
                       contents: List[str],
                       source_language_code=SOURCE_LANG,
                       target_language_code=TARGET_LANG,
                       location=LOCATION) -> List[str]:
    """调用 v3 translate_text 批量翻译（统一反转义 HTML 实体）"""
    parent = f"projects/{project_id}/locations/{location}"
    # 调试可打开：
    # print(f"[DEBUG] parent = {parent}")
    # print(f"[DEBUG] batch size = {len(contents)}; sample = {contents[:2]}")
    resp = client.translate_text(
        request={
            "parent": parent,
            "contents": contents,
            "mime_type": "text/plain",
            "source_language_code": source_language_code,
            "target_language_code": target_language_code,
        }
    )
    return [html.unescape(t.translated_text) for t in resp.translations]

def translate_all_v3(client, project_id, lines: List[str]) -> List[str]:
    out = []
    for chunk in batch_iter(lines):
        out.extend(translate_batch_v3(client, project_id, chunk))
        time.sleep(SLEEP_BETWEEN_BATCH)
    return out


In [69]:
def srt_pinyin_top_ja_bottom(input_srt: str,
                             output_srt: str,
                             project_id: str = PROJECT_ID,
                             tone_type=TONE_TYPE,
                             source=SOURCE_LANG,
                             target=TARGET_LANG,
                             location=LOCATION):
    """读取 SRT → 抽取对白 → v3 翻译成日语 → 拼音(上)+日语(下)回写 → 输出新 SRT"""
    text = Path(input_srt).read_text(encoding="utf-8", errors="ignore")
    raw_blocks = text.strip().split("\n\n")

    # 收集对白
    idx_map: List[Tuple[int,int,str]] = []
    to_translate: List[str] = []
    for bi, block in enumerate(raw_blocks):
        lines = block.splitlines()
        for li, line in enumerate(lines):
            if is_dialogue(line):
                raw = clean_text(line)
                if raw:
                    idx_map.append((bi, li, raw))
                    to_translate.append(raw)

    if not to_translate:
        print("未检测到可翻译对白；请检查 SRT 是否标准。")
        return

    client = translate_v3.TranslationServiceClient()

    # 翻译
    ja_all = translate_all_v3(client, project_id, to_translate)

    # 回写：拼音在上，日语在下
    new_blocks = [b.splitlines() for b in raw_blocks]
    for (bi, li, raw), ja in zip(idx_map, ja_all):
        py = to_pinyin(raw, tone_type=tone_type)
        new_blocks[bi][li] = f"{py}\n{ja}"

    Path(output_srt).write_text("\n\n".join("\n".join(x) for x in new_blocks), encoding="utf-8")
    print("✅ 完成，输出：", output_srt)


In [70]:
# 把下面的 input_srt 改成 Step 3 上传显示出来的文件名
input_srt = list(uploaded.keys())[0]     # 或直接写 "movie.srt"
output_srt = "movie_pinyin_ja.srt"

srt_pinyin_top_ja_bottom(input_srt, output_srt, project_id=PROJECT_ID)

# 下载结果
from google.colab import files
files.download(output_srt)


✅ 完成，输出： movie_pinyin_ja.srt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>